In [ ]:
from google.colab import drive
import os

# Montar Google Drive
drive.mount('/content/drive')

# Ruta base del proyecto en Drive
rutaBase = '/content/drive/MyDrive/SGB/datos/'
os.makedirs(rutaBase, exist_ok=True)

# Rutas de cada archivo de persistencia
rutaUsuarios   = rutaBase + 'usuarios.txt'
rutaLibros     = rutaBase + 'libros.txt'
rutaCategorias = rutaBase + 'categorias.txt'
rutaPrestamos  = rutaBase + 'prestamos.txt'
rutaMultas     = rutaBase + 'multas.txt'
rutaReservas   = rutaBase + 'reservas.txt'

In [ ]:
from datetime import date, timedelta

class RepositorioArchivo:
    def __init__(self, rutaArchivo):
        self.rutaArchivo = rutaArchivo
        if not os.path.exists(self.rutaArchivo):
            open(self.rutaArchivo, "w").close()

    def leerTodo(self):
        listaRegistros = []
        with open(self.rutaArchivo, "r", encoding="utf-8") as archivo:
            for linea in archivo:
                linea = linea.strip()
                if linea:
                    listaRegistros.append(linea.split("|"))
        return listaRegistros

    def guardarTodo(self, listaRegistros):
        with open(self.rutaArchivo, "w", encoding="utf-8") as archivo:
            for registro in listaRegistros:
                archivo.write("|".join(registro) + "\n")

    def agregar(self, registro):
        with open(self.rutaArchivo, "a", encoding="utf-8") as archivo:
            archivo.write("|".join(registro) + "\n")

    def siguienteId(self):
        return str(len(self.leerTodo()) + 1)

In [ ]:
class ServicioUsuarios:
    tiposValidos = ["estudiante", "docente", "externo"]

    def __init__(self, repoUsuarios):
        self.repoUsuarios = repoUsuarios

    def consultar(self, documento):
        for usuario in self.repoUsuarios.leerTodo():
            if usuario[0] == documento:
                return usuario
        return None

    def registrar(self, documento, nombre, tipo, correo, telefono):
        if self.consultar(documento) is not None:
            return False, "Ya existe un usuario con ese documento"
        if tipo not in self.tiposValidos:
            return False, "Tipo inválido (estudiante, docente o externo)"
        self.repoUsuarios.agregar([documento, nombre, tipo, correo, telefono, "activo"])
        return True, "Usuario registrado"

    def actualizar(self, documento, correo, telefono, estado):
        listaUsuarios = self.repoUsuarios.leerTodo()
        for usuario in listaUsuarios:
            if usuario[0] == documento:
                if correo != "":
                    usuario[3] = correo
                if telefono != "":
                    usuario[4] = telefono
                if estado in ("activo", "suspendido"):
                    usuario[5] = estado
                self.repoUsuarios.guardarTodo(listaUsuarios)
                return True, "Usuario actualizado"
        return False, "Usuario no encontrado"

    def listar(self):
        return self.repoUsuarios.leerTodo()

In [ ]:
class ServicioLibros:
    def __init__(self, repoLibros, repoCategorias):
        self.repoLibros = repoLibros
        self.repoCategorias = repoCategorias

    def registrarCategoria(self, nombre):
        for categoria in self.repoCategorias.leerTodo():
            if categoria[0].lower() == nombre.lower():
                return False, "La categoría ya existe"
        self.repoCategorias.agregar([nombre])
        return True, "Categoría registrada"

    def listarCategorias(self):
        return self.repoCategorias.leerTodo()

    def registrarLibro(self, isbn, titulo, autor, categoria, anio, ejemplares):
        if self.consultarPorIsbn(isbn) is not None:
            return False, "Ya existe un libro con ese ISBN"
        nombresCategorias = [c[0] for c in self.repoCategorias.leerTodo()]
        if categoria not in nombresCategorias:
            return False, "La categoría no existe; regístrala primero"
        self.repoLibros.agregar([isbn, titulo, autor, categoria, anio, ejemplares, ejemplares])
        return True, "Libro registrado"

    def consultarPorIsbn(self, isbn):
        for libro in self.repoLibros.leerTodo():
            if libro[0] == isbn:
                return libro
        return None

    def buscarPorTitulo(self, texto):
        return [l for l in self.repoLibros.leerTodo() if texto.lower() in l[1].lower()]

    def buscarPorCategoria(self, categoria):
        return [l for l in self.repoLibros.leerTodo() if l[3].lower() == categoria.lower()]

    def listarDisponibles(self):
        return [l for l in self.repoLibros.leerTodo() if int(l[6]) > 0]

    def cambiarDisponibles(self, isbn, cambio):
        listaLibros = self.repoLibros.leerTodo()
        for libro in listaLibros:
            if libro[0] == isbn:
                libro[6] = str(int(libro[6]) + cambio)
        self.repoLibros.guardarTodo(listaLibros)

In [ ]:
class ServicioMultas:
    valorMultaDia = 1000  # COP por día de retraso

    def __init__(self, repoMultas):
        self.repoMultas = repoMultas

    def generar(self, documento, idPrestamo, diasRetraso):
        monto = diasRetraso * self.valorMultaDia
        self.repoMultas.agregar([self.repoMultas.siguienteId(), documento, idPrestamo, str(monto), "PENDIENTE"])
        return monto

    def listarPendientes(self, documento):
        return [m for m in self.repoMultas.leerTodo() if m[1] == documento and m[4] == "PENDIENTE"]

    def pagar(self, idMulta):
        listaMultas = self.repoMultas.leerTodo()
        for multa in listaMultas:
            if multa[0] == idMulta:
                if multa[4] == "PAGADA":
                    return False, "Esa multa ya estaba pagada"
                multa[4] = "PAGADA"
                self.repoMultas.guardarTodo(listaMultas)
                return True, "Multa pagada"
        return False, "Multa no encontrada"

In [ ]:
class ServicioPrestamos:
    diasPorTipo = {"estudiante": 7, "docente": 15, "externo": 5}
    cupoPorTipo = {"estudiante": 3, "docente": 5, "externo": 2}

    def __init__(self, repoPrestamos, servicioUsuarios, servicioLibros, servicioMultas):
        self.repoPrestamos = repoPrestamos
        self.servicioUsuarios = servicioUsuarios
        self.servicioLibros = servicioLibros
        self.servicioMultas = servicioMultas

    def prestar(self, documento, isbn, fechaTexto):
        usuario = self.servicioUsuarios.consultar(documento)
        if usuario is None:
            return False, "Usuario no encontrado"
        if usuario[5] != "activo":
            return False, "El usuario está suspendido"
        if len(self.servicioMultas.listarPendientes(documento)) > 0:
            return False, "El usuario tiene multas pendientes"

        tipoUsuario = usuario[2]
        prestamosActivos = [p for p in self.listarPorUsuario(documento) if p[6] == "ACTIVO"]
        if len(prestamosActivos) >= self.cupoPorTipo[tipoUsuario]:
            return False, "El usuario ya alcanzó su cupo de préstamos"

        libro = self.servicioLibros.consultarPorIsbn(isbn)
        if libro is None:
            return False, "Libro no encontrado"
        if int(libro[6]) <= 0:
            return False, "No hay ejemplares disponibles (puedes reservarlo)"

        fechaPrestamo = date.fromisoformat(fechaTexto)
        fechaLimite = fechaPrestamo + timedelta(days=self.diasPorTipo[tipoUsuario])
        self.servicioLibros.cambiarDisponibles(isbn, -1)
        self.repoPrestamos.agregar([self.repoPrestamos.siguienteId(), documento, isbn,
                                    str(fechaPrestamo), str(fechaLimite), "", "ACTIVO"])
        return True, "Préstamo registrado. Fecha límite: " + str(fechaLimite)

    def devolver(self, idPrestamo, fechaTexto):
        listaPrestamos = self.repoPrestamos.leerTodo()
        for prestamo in listaPrestamos:
            if prestamo[0] == idPrestamo and prestamo[6] == "ACTIVO":
                fechaDevolucion = date.fromisoformat(fechaTexto)
                prestamo[5] = str(fechaDevolucion)
                prestamo[6] = "DEVUELTO"
                self.repoPrestamos.guardarTodo(listaPrestamos)
                self.servicioLibros.cambiarDisponibles(prestamo[2], 1)

                diasRetraso = (fechaDevolucion - date.fromisoformat(prestamo[4])).days
                if diasRetraso > 0:
                    monto = self.servicioMultas.generar(prestamo[1], prestamo[0], diasRetraso)
                    return True, "Devuelto con " + str(diasRetraso) + " día(s) de retraso. Multa: COP " + str(monto)
                return True, "Devolución registrada sin retraso"
        return False, "Préstamo no encontrado o ya devuelto"

    def listarPorUsuario(self, documento):
        return [p for p in self.repoPrestamos.leerTodo() if p[1] == documento]

    def listarVencidos(self, fechaTexto):
        return [p for p in self.repoPrestamos.leerTodo() if p[6] == "ACTIVO" and p[4] < fechaTexto]

    def historial(self, documento):
        listaHistorial = self.listarPorUsuario(documento)
        listaHistorial.sort(key=lambda p: p[3])  # orden cronológico
        return listaHistorial

In [ ]:
class ServicioReservas:
    def __init__(self, repoReservas, servicioUsuarios, servicioLibros):
        self.repoReservas = repoReservas
        self.servicioUsuarios = servicioUsuarios
        self.servicioLibros = servicioLibros

    def reservar(self, documento, isbn):
        if self.servicioUsuarios.consultar(documento) is None:
            return False, "Usuario no encontrado"
        libro = self.servicioLibros.consultarPorIsbn(isbn)
        if libro is None:
            return False, "Libro no encontrado"
        if int(libro[6]) > 0:
            return False, "El libro tiene ejemplares disponibles; solicita el préstamo"
        self.repoReservas.agregar([self.repoReservas.siguienteId(), documento, isbn, str(date.today()), "ACTIVA"])
        return True, "Reserva registrada"

    def cancelar(self, idReserva):
        listaReservas = self.repoReservas.leerTodo()
        for reserva in listaReservas:
            if reserva[0] == idReserva and reserva[4] == "ACTIVA":
                reserva[4] = "CANCELADA"
                self.repoReservas.guardarTodo(listaReservas)
                return True, "Reserva cancelada"
        return False, "Reserva no encontrada o ya cancelada"

    def listarPorUsuario(self, documento):
        return [r for r in self.repoReservas.leerTodo() if r[1] == documento]

In [ ]:
servicioUsuarios  = ServicioUsuarios(RepositorioArchivo(rutaUsuarios))
servicioLibros    = ServicioLibros(RepositorioArchivo(rutaLibros), RepositorioArchivo(rutaCategorias))
servicioMultas    = ServicioMultas(RepositorioArchivo(rutaMultas))
servicioPrestamos = ServicioPrestamos(RepositorioArchivo(rutaPrestamos), servicioUsuarios, servicioLibros, servicioMultas)
servicioReservas  = ServicioReservas(RepositorioArchivo(rutaReservas), servicioUsuarios, servicioLibros)
print("Servicios listos")

In [ ]:
import ipywidgets as widgets
from IPython.display import display

def cargarDatosPrueba():
    RepositorioArchivo(rutaUsuarios).guardarTodo([
        ["1001", "Ana Pérez", "estudiante", "ana@correo.com", "3001234567", "activo"],
        ["1002", "Luis Gómez", "docente", "luis@correo.com", "3109876543", "activo"]
    ])
    RepositorioArchivo(rutaCategorias).guardarTodo([
        ["Programación"],
        ["Novela"]
    ])
    RepositorioArchivo(rutaLibros).guardarTodo([
        ["978-1", "Python Básico", "Ana Torres", "Programación", "2022", "3", "2"],
        ["978-2", "Cien años de soledad", "Gabriel García Márquez", "Novela", "1967", "1", "0"]
    ])
    RepositorioArchivo(rutaPrestamos).guardarTodo([
        ["1", "1001", "978-1", "2026-10-20", "2026-10-27", "", "ACTIVO"],
        ["2", "1002", "978-2", "2026-11-01", "2026-11-16", "", "ACTIVO"],
        ["3", "1001", "978-1", "2026-09-01", "2026-09-08", "2026-09-10", "DEVUELTO"]
    ])
    RepositorioArchivo(rutaMultas).guardarTodo([
        ["1", "1001", "3", "2000", "PENDIENTE"]
    ])
    RepositorioArchivo(rutaReservas).guardarTodo([
        ["1", "1001", "978-2", "2026-11-02", "ACTIVA"]
    ])
    print("Datos de prueba cargados en:", rutaBase)
    for nombreArchivo in os.listdir(rutaBase):
        print(" -", nombreArchivo)

botonPrueba = widgets.Button(description="Cargar datos de prueba", button_style="info")
salidaPrueba = widgets.Output()

def alPresionarBoton(boton):
    with salidaPrueba:
        salidaPrueba.clear_output()
        cargarDatosPrueba()

botonPrueba.on_click(alPresionarBoton)
display(botonPrueba, salidaPrueba)

In [ ]:
def mostrar(listaRegistros):
    if len(listaRegistros) == 0:
        print("(sin registros)")
    for registro in listaRegistros:
        print(" | ".join(registro))

def pedirFecha(mensaje):
    textoFecha = input(mensaje + " (AAAA-MM-DD, vacío = hoy): ").strip()
    if textoFecha == "":
        return str(date.today())
    return textoFecha

In [ ]:
opcionUsuarios = ""
while opcionUsuarios != "0":
    print("\n--- USUARIOS ---")
    print("1. Registrar usuario")
    print("2. Consultar por documento")
    print("3. Actualizar contacto / estado")
    print("4. Listar usuarios")
    print("0. Volver")
    opcionUsuarios = input("Opción: ")

    if opcionUsuarios == "1":
        documento = input("Documento: ")
        nombre = input("Nombre completo: ")
        tipo = input("Tipo (estudiante/docente/externo): ")
        correo = input("Correo: ")
        telefono = input("Teléfono: ")
        print(servicioUsuarios.registrar(documento, nombre, tipo, correo, telefono)[1])

    elif opcionUsuarios == "2":
        usuarioEncontrado = servicioUsuarios.consultar(input("Documento: "))
        mostrar([usuarioEncontrado] if usuarioEncontrado else [])

    elif opcionUsuarios == "3":
        documento = input("Documento: ")
        correo = input("Nuevo correo (vacío = no cambiar): ")
        telefono = input("Nuevo teléfono (vacío = no cambiar): ")
        estado = input("Estado (activo/suspendido, vacío = no cambiar): ")
        print(servicioUsuarios.actualizar(documento, correo, telefono, estado)[1])

    elif opcionUsuarios == "4":
        mostrar(servicioUsuarios.listar())

In [ ]:
opcionLibros = ""
while opcionLibros != "0":
    print("\n--- LIBROS Y CATEGORÍAS ---")
    print("1. Registrar categoría")
    print("2. Listar categorías")
    print("3. Registrar libro")
    print("4. Consultar por ISBN")
    print("5. Buscar por título")
    print("6. Buscar por categoría")
    print("7. Listar libros disponibles")
    print("0. Volver")
    opcionLibros = input("Opción: ")

    if opcionLibros == "1":
        print(servicioLibros.registrarCategoria(input("Nombre de la categoría: "))[1])

    elif opcionLibros == "2":
        mostrar(servicioLibros.listarCategorias())

    elif opcionLibros == "3":
        isbn = input("ISBN: ")
        titulo = input("Título: ")
        autor = input("Autor: ")
        categoria = input("Categoría: ")
        anio = input("Año de publicación: ")
        ejemplares = input("Número de ejemplares: ")
        print(servicioLibros.registrarLibro(isbn, titulo, autor, categoria, anio, ejemplares)[1])

    elif opcionLibros == "4":
        libroEncontrado = servicioLibros.consultarPorIsbn(input("ISBN: "))
        mostrar([libroEncontrado] if libroEncontrado else [])

    elif opcionLibros == "5":
        mostrar(servicioLibros.buscarPorTitulo(input("Título (o parte): ")))

    elif opcionLibros == "6":
        mostrar(servicioLibros.buscarPorCategoria(input("Categoría: ")))

    elif opcionLibros == "7":
        mostrar(servicioLibros.listarDisponibles())

In [ ]:
opcionPrestamos = ""
while opcionPrestamos != "0":
    print("\n--- PRÉSTAMOS ---")
    print("1. Registrar préstamo")
    print("2. Registrar devolución")
    print("3. Listar préstamos de un usuario")
    print("4. Listar préstamos vencidos a una fecha")
    print("0. Volver")
    opcionPrestamos = input("Opción: ")

    if opcionPrestamos == "1":
        documento = input("Documento del usuario: ")
        isbn = input("ISBN del libro: ")
        fechaPrestamo = pedirFecha("Fecha del préstamo")
        print(servicioPrestamos.prestar(documento, isbn, fechaPrestamo)[1])

    elif opcionPrestamos == "2":
        idPrestamo = input("Id del préstamo: ")
        fechaDevolucion = pedirFecha("Fecha de devolución")
        print(servicioPrestamos.devolver(idPrestamo, fechaDevolucion)[1])

    elif opcionPrestamos == "3":
        mostrar(servicioPrestamos.listarPorUsuario(input("Documento del usuario: ")))

    elif opcionPrestamos == "4":
        mostrar(servicioPrestamos.listarVencidos(pedirFecha("Fecha de corte")))

In [ ]:
opcionExtras = ""
while opcionExtras != "0":
    print("\n--- RESERVAS, MULTAS E HISTORIAL ---")
    print("1. Reservar libro sin ejemplares")
    print("2. Cancelar reserva")
    print("3. Consultar multas pendientes de un usuario")
    print("4. Registrar pago de una multa")
    print("5. Historial de préstamos de un usuario")
    print("0. Volver")
    opcionExtras = input("Opción: ")

    if opcionExtras == "1":
        documento = input("Documento del usuario: ")
        isbn = input("ISBN del libro: ")
        print(servicioReservas.reservar(documento, isbn)[1])

    elif opcionExtras == "2":
        print(servicioReservas.cancelar(input("Id de la reserva: "))[1])

    elif opcionExtras == "3":
        mostrar(servicioMultas.listarPendientes(input("Documento del usuario: ")))

    elif opcionExtras == "4":
        print(servicioMultas.pagar(input("Id de la multa: "))[1])

    elif opcionExtras == "5":
        mostrar(servicioPrestamos.historial(input("Documento del usuario: ")))